# Optional: 2D OPFData, Algorithm 1 versus residual-based PHD

Run in Google Colab with the small CSV in mounted Drive. Change the `CSV` path as needed. Both estimates use the same observations and original units.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone --depth=1 -q https://github.com/hitotoro24/PICART.git
import sys
sys.path.insert(0, '/content/PICART/weeks/2026-W46/cedar_baseline')
from algorithm1_tree_hessian import estimate_tree_hessian


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.tree import DecisionTreeRegressor

CSV = Path('/content/drive/MyDrive/118_ieee_smol.csv')  # change this path
frame = pd.read_csv(CSV, header=None, usecols=[0, 1, 2])
y, X = frame.iloc[:,0].to_numpy(), frame.iloc[:,1:].to_numpy()
Z = X-X.mean(0); D = np.c_[np.ones(len(y)),Z]
r = y-D@np.linalg.lstsq(D,y,rcond=None)[0]
S, M = Z.T@Z/len(y), Z.T@(r[:,None]*Z)/len(y)
A = np.linalg.solve(S,M); PHD = np.linalg.solve(S,A.T).T
PHD = (PHD+PHD.T)/2
Q, raw, info = estimate_tree_hessian(DecisionTreeRegressor(max_depth=7, random_state=0).fit(X,y), X)
print('PHD:\n', PHD, '\nAlgorithm 1:\n', Q if Q is not None else 'not identified')
print('Raw row matrix:\n',raw,'\n(axis, nodes, pairs, rank):',info)
